# When Nearly Rank-One Predictions Require Large Sequential Models
**Vashista Nobaub · Datar Consulting**

Run this notebook in a fresh Google Colab CPU runtime using **Runtime → Run all**. It regenerates both vector figures, the numerical tables, the paper, the standalone mathematical supplement, the anonymous review PDF and the source ZIP to upload to arXiv. All TeX sources and eight original measurements travel with the notebook. Repository access is unnecessary.

The default mode validates the deposited scores and rebuilds every output. An optional inference replay downloads about 757 MiB and runs the frozen four-bit Qwen2.5 checkpoint on CPU. The final cell downloads the arXiv ZIP in Colab.

## Choose the replay mode
Enable the first switch to repeat model inference and compare its outputs with the deposition. The source archive always uses the frozen scores so that platform-specific timing cannot change a result in the article.

In [ ]:
RERUN_CHECKPOINT = False  # @param {type:"boolean"}
DOWNLOAD_ARXIV_ZIP = True  # @param {type:"boolean"}

## Set up the runtime
The notebook writes to a fresh directory. It installs missing scientific Python packages and, on a fresh Colab Linux image, the required TeX and Poppler packages. Initial setup may take several minutes.

In [ ]:
import importlib.util, os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path
ROOT = Path(tempfile.mkdtemp(prefix="draga_paper_"))
for name in ("paper/figures", "results", "experiments", "artifacts"):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
missing = [p for p in ("numpy","scipy","matplotlib") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,"-m","pip","install","-q",*missing],check=True)
if not (shutil.which("pdflatex") and shutil.which("pdftoppm") and shutil.which("kpsewhich")
        and subprocess.run(["kpsewhich","lineno.sty"],capture_output=True).returncode == 0):
    if not shutil.which("apt-get") or os.geteuid() != 0:
        raise RuntimeError("Install TeX Live LaTeX extra, fonts recommended, Latin Modern and Poppler, then rerun.")
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","texlive-latex-extra","texlive-fonts-recommended",
                    "lmodern","poppler-utils"],check=True)
print("Working folder:",ROOT,"Python:",platform.python_version())

## Restore the text inputs
The compact embedded package contains the paper sources, the official NeurIPS style, the completed review checklist and recorded model scores. Its checksum is verified. It contains no figures, PDFs or model parameters. The executable methods are visible in the subsequent cells.

In [ ]:
# @title Restore checksum-verified inputs
import base64, hashlib, io, zipfile, json
encoded = """
...
AAAAAAwADAAcAwAAXn4AAAAA
"""
bundle = base64.b64decode(encoded)
assert hashlib.sha256(bundle).hexdigest() == "c43be83f32a1609886811dc9029849fb5fd73fa475703a92b2b153ab6aa9fa88"
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert member.startswith(("paper/", "results/")) and ".." not in Path(member).parts
    archive.extractall(ROOT)
print("Restored",len(archive.namelist()),"verified files.")

## Define the teacher and the comparator
Each gate task has one successful action sequence. The next cell writes the actual scoring program. Each score includes the final stop token. The constrained policy renormalizes the same three token IDs after every prefix. The program also tracks the full-vocabulary probability. The autonomous controller cannot read back the prompt or access an external time counter.

In [ ]:
os.chdir(ROOT)

In [ ]:
%%writefile experiments/planner.py
"""Teacher-forced policy scoring on the complete binary gate-suite. CPU ONNX."""
from pathlib import Path
from itertools import product
import argparse, hashlib, json, math, platform, time, importlib.metadata
import numpy as np
from scipy.special import logsumexp, xlogy

MODEL = "onnx-community/Qwen2.5-0.5B-Instruct"
REVISION = "22942cb7d7ba4cc81bb4673549ca4d4614469b5e"
DIGESTS = {"model_q4.onnx": "09235a3b1c135cd04ef570e5053b5c079e028078a2cc5f76ba6a251e91bf3296",
           "tokenizer.json": "a8506e7111b80c6d8635951a02eab0f4e1a8e4e5772da83846579e97b16f61bf"}

def digest(path):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def instance(m, descending):
    order = list(range(2**m))
    if descending:
        order.reverse()
    target = "".join(format(i, f"0{m}b") for i in order)
    direction = "descending" if descending else "ascending"
    prompt = (f"The robot must unlock all {2**m} gates in {direction} numerical order, "
              f"starting at gate {order[0]} and ending at gate {order[-1]}. "
              f"To unlock gate j, encode j as exactly {m} binary digits, most significant first "
              "(pad with leading zeros). For digit 0 send A; for digit 1 send B. "
              "A wrong action fails the mission. Output only the actions separated by spaces, "
              "then end the response immediately after the last gate. Do not print gate numbers.")
    return prompt, tuple(map(int, target))

def verify(actions, m, descending):
    if len(actions) != m * 2**m: return False
    rooms = [int("".join(str(a) for a in actions[j:j+m]), 2)
             for j in range(0, len(actions), m)]
    return rooms == sorted(range(2**m), reverse=descending)

def lower_kl(logp, logcap):
    if logcap >= 0 or logp <= logcap: return 0.0
    p, c = math.exp(logp), math.exp(logcap)
    return float(p*(logp-logcap) + xlogy(1-p, 1-p) - (1-p)*math.log1p(-c))

def bounds(m):
    d, K, L = 2, 2**m, m*2**m
    # The scalar bound includes termination and letter frequencies.
    one = L*(math.log(L)-math.log(L+1)-math.log(d)) - math.log(L+1)
    return {str(n): (one if n==1 else min(0., K*math.log(n/K))) for n in (1,2,4)}

class Reader:
    def __init__(self, model, tokenizer):
        import onnxruntime as ort
        from tokenizers import Tokenizer
        self.tokenizer=Tokenizer.from_file(str(tokenizer))
        options=ort.SessionOptions()
        options.intra_op_num_threads=3
        options.inter_op_num_threads=1
        self.session=ort.InferenceSession(str(model),sess_options=options,providers=["CPUExecutionProvider"])
        self.tokens=[self.tokenizer.encode(t,add_special_tokens=False).ids for t in (" A"," B")]
        assert all(len(t)==1 for t in self.tokens)
        self.tokens=[t[0] for t in self.tokens]+[self.tokenizer.token_to_id("<|im_end|>")]
        assert len(set(self.tokens))==3 and None not in self.tokens

    def logprobs(self, prompt, actions):
        text=("<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
              "<|im_start|>user\n"+prompt+"<|im_end|>\n<|im_start|>assistant\n")
        prefix=self.tokenizer.encode(text,add_special_tokens=False).ids
        seq=prefix+[self.tokens[a] for a in actions]
        ids=np.asarray([seq],dtype=np.int64)
        feed={"input_ids":ids,"position_ids":np.arange(len(seq),dtype=np.int64)[None,:],
              "attention_mask":np.ones_like(ids)}
        for inp in self.session.get_inputs():
            if inp.name.startswith("past_key_values."):
                feed[inp.name]=np.empty((1,2,0,64),np.float32)
        logits=self.session.run(["logits"],feed)[0][0,len(prefix)-1:].astype(np.float64)
        selected=logits[:,self.tokens]
        return selected-logsumexp(selected,axis=1)[:,None], selected-logsumexp(logits,axis=1)[:,None]

def capped_hankel(reader,prompt,T=3):
    transitions={}
    for path in product(range(2),repeat=T-1):
        restricted,_=reader.logprobs(prompt,path)
        for j in range(len(path)+1):
            transitions.setdefault(path[:j],np.exp(restricted[j]))
    words=[p for j in range(T+1) for p in product(range(2),repeat=j)]
    mass={}
    for w in words:
        v=math.prod(transitions[w[:j]][a] for j,a in enumerate(w))
        mass[w]=v*(transitions[w][2] if len(w)<T else 1.0)
    assert abs(sum(mass.values())-1)<1.e-5
    h=np.array([[mass.get(u+v,0.) for v in words] for u in words])
    sv=np.linalg.svd(h,compute_uv=False)
    skew=np.linalg.norm(h-h.T,ord=2)/2
    return {"horizon":T,"words":{"".join(map(str,w)):float(p) for w,p in mass.items()},
            "sigma1":float(sv[0]),"sigma2":float(sv[1]),"skew_floor":float(skew),
            "ratio":float(skew/sv[1])}

def run(model,tokenizer,out):
    assert digest(model)==DIGESTS["model_q4.onnx"] and digest(tokenizer)==DIGESTS["tokenizer.json"]
    reader=Reader(model,tokenizer)
    rows=[]; t=time.monotonic()
    for m in (1,2,3,4):
        for desc in (False,True):
            prompt,actions=instance(m,desc)
            assert verify(actions,m,desc)
            cond,raw=reader.logprobs(prompt,actions)
            target=list(actions)+[2]
            logp=float(cond[np.arange(len(target)),target].sum())
            rawp=float(raw[np.arange(len(target)),target].sum())
            row={"m":m,"order":"descending" if desc else "ascending","prompt":prompt,
                 "target":"".join("AB"[a] for a in actions),"logp":logp,"raw_logp":rawp,
                 "greedy_success":bool(np.array_equal(np.argmax(cond,axis=1),target)),
                 "logcaps":bounds(m),"kl_floor":{k:lower_kl(logp,v) for k,v in bounds(m).items()},
                 "step_logp":cond[np.arange(len(target)),target].tolist()}
            if m==1: row["capped"]=capped_hankel(reader,prompt)
            rows.append(row)
            print(m,row["order"],"ln P =",round(logp,4),"KL",row["kl_floor"],flush=True)
    payload={"model":MODEL,"revision":REVISION,"sha256":DIGESTS,"protocol_tokens":reader.tokens,
             "threads":3,"platform":platform.platform(),"seconds":time.monotonic()-t,
             "versions":{p:importlib.metadata.version(p) for p in ["onnxruntime","tokenizers","numpy","scipy"]},
             "instances":rows}
    Path(out).write_text(json.dumps(payload,indent=2)+"\n")

if __name__=="__main__":
    p=argparse.ArgumentParser(); p.add_argument("model");p.add_argument("tokenizer");p.add_argument("output")
    a=p.parse_args(); run(a.model,a.tokenizer,a.output)

## Optional checkpoint replay
The immutable revision and SHA-256 digests are in the scoring code above. Replaying compares all step scores, event certificates, the verifier result and the enumerated short-horizon distributions. Thresholds are numerical tolerances on exported CPU inference. A mismatch raises an informative assertion.

In [ ]:
import numpy as np
sys.path.insert(0,str(ROOT/"experiments"))
from planner import MODEL, REVISION, DIGESTS, digest
frozen=json.loads((ROOT/"results/qwen.json").read_text())
if RERUN_CHECKPOINT:
    import urllib.request
    for pkg in ("onnxruntime","tokenizers"):
        if importlib.util.find_spec(pkg) is None:
            subprocess.run([sys.executable,"-m","pip","install","-q",pkg],check=True)
    weights=ROOT/"weights"
    weights.mkdir(exist_ok=True)
    for name,remote in [("model_q4.onnx","onnx/model_q4.onnx"),("tokenizer.json","tokenizer.json")]:
        target=weights/name
        if not target.exists() or digest(target)!=DIGESTS[name]:
            part=target.with_suffix(target.suffix+".part")
            urllib.request.urlretrieve(f"https://huggingface.co/{MODEL}/resolve/{REVISION}/{remote}",part)
            if digest(part)!=DIGESTS[name]: raise ValueError("Checksum mismatch: "+name)
            part.replace(target)
    from planner import run
    replay=ROOT/"results/qwen-replayed.json"
    run(weights/"model_q4.onnx",weights/"tokenizer.json",replay)
    current=json.loads(replay.read_text())
    assert len(current["instances"])==len(frozen["instances"])==8
    for a,b in zip(frozen["instances"],current["instances"]):
        assert (a["target"],a["prompt"],a["greedy_success"])==(b["target"],b["prompt"],b["greedy_success"])
        for key in ("logp","raw_logp","step_logp"):
            np.testing.assert_allclose(a[key],b[key],atol=2e-3,rtol=1e-5,err_msg=f'{a["order"]}/{a["m"]}/{key}')
        for key in a["kl_floor"]:
            np.testing.assert_allclose(a["kl_floor"][key],b["kl_floor"][key],atol=2e-3)
        if "capped" in a:
            for word,prob in a["capped"]["words"].items():
                np.testing.assert_allclose(prob,b["capped"]["words"][word],atol=3e-5)
    print("All eight checkpoint instances replayed.")
else:
    print("Using eight recorded measurements from",MODEL,REVISION)

## Reproduce the mathematical and empirical figures
The first panel tests finite Hankel identities and plots certified inequalities. The second gives the limiting state–log-loss frontier and finite intervals. The group controller is checked by explicit forward multiplication, summing all its paths, for every small block width and retained prefix length. The checkpoint plot compares each completed plan with the one-state probability cap. All eight measurements remain visible, including inconclusive cases.

In [ ]:
%%writefile experiments/figures.py
"""Generate both vector figures and all empirical tables from recorded inputs."""
from pathlib import Path
from itertools import product
from math import comb, log, exp
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import xlogy
from planner import instance, verify, bounds, lower_kl

def check_small_cases():
    # Verify the exact cut formula entrywise on every word through length four.
    words=["".join(v) for j in range(5) for v in product("ab",repeat=j)]
    for k in (1,2,3):
        r=.75; q=1-r*r
        def zword(w,j):
            start=w[:-j] if j else w
            return np.sqrt(q)*r**len(start) if w==start+"b"*j and set(start)<={"a"} else 0.
        z=np.array([[zword(w,j) for w in words] for j in range(k+1)])
        rhs=np.outer(z[0],z[k])
        for j in range(k): rhs[:,words.index("b"*j)] += np.sqrt(q)*z[k-j]
        def source(w):
            return q*r**(len(w)-k) if w.endswith("b"*k) and set(w[:-k])<={"a"} else 0.
        assert np.allclose(rhs,[[source(u+v) for v in words] for u in words],atol=1e-14)
    rng=np.random.default_rng(705)
    for n in (1,2,3,4):
        for m in (1,2,3):
            raw=rng.uniform(.1,1,size=(n,2*n+1))
            raw/=raw.sum(axis=1,keepdims=True)
            ts=[raw[:,j*n:(j+1)*n] for j in range(2)]
            _,path=instance(m,False); row=np.ones(n)/n
            for a in path: row=row@ts[a]
            cap=bounds(m)
            generic=min(1.,(n/2**m)**(2**m))
            assert float(row@raw[:,-1]) <= generic+1e-13
            if n==1: assert float(row@raw[:,-1]) <= exp(cap["1"])+1e-13

def check_group_controller():
    # Enumerate all state paths by a forward pass. No simulated trajectories.
    for m in range(1,5):
        for ell in range(m+1):
            G,D=2**ell,2**(m-ell)
            n=m*G+1; terminal=n-1
            mats=np.zeros((2,n,n))
            for g in range(G):
                for j in range(m):
                    src=g*m+j
                    actions=([((g>>(ell-j-1))&1,1.)] if j<ell else [(0,.5),(1,.5)])
                    for a,p in actions:
                        if j+1<m: mats[a,src,src+1]+=p
                        else:
                            mats[a,src,g*m]+=p*(1-1/D)
                            mats[a,src,(g+1)*m]+=p/D
            assert np.allclose(mats.sum(axis=(0,2))[:-1],1)
            state=np.eye(1,n)[0]
            for a in instance(m,False)[1]: state=state@mats[a]
            path_cost=(m-ell)*2**m*log(2)+G*log(D)
            if D>1: path_cost-=G*(D-1)*np.log1p(-1/D)
            assert np.isclose(-log(state[terminal]),path_cost,atol=1e-10)

def generate(root):
    root=Path(root)
    figures=root/"paper/figures"; figures.mkdir(parents=True,exist_ok=True)
    paper=root/"paper"
    rec=json.loads((root/"results/qwen.json").read_text()); rows=rec["instances"]
    assert [(r["m"],r["order"]) for r in rows]==[(m,o) for m in range(1,5) for o in ("ascending","descending")]
    check_small_cases()
    check_group_controller()
    for row in rows:
        prompt,w=instance(row["m"],row["order"]=="descending")
        assert row["prompt"]==prompt and verify(w,row["m"],row["order"]=="descending")
        assert row["target"]=="".join("AB"[a] for a in w)
        assert abs(sum(row["step_logp"])-row["logp"])<1e-9
        for n,c in bounds(row["m"]).items():
            assert abs(c-row["logcaps"][n])<1e-12
            assert abs(lower_kl(row["logp"],c)-row["kl_floor"][n])<1e-12
        if "capped" in row:
            data=row["capped"]; mass=data["words"];assert abs(sum(mass.values())-1.)<1.e-6
            mat=np.array([[mass.get(a+b,0.) for b in mass] for a in mass])
            assert np.isclose(np.linalg.svd(mat,compute_uv=False)[1],data["sigma2"])
            assert np.isclose(np.linalg.norm(mat-mat.T,2)/2,data["skew_floor"])
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9,"axes.spines.top":False,
                         "axes.spines.right":False,"pdf.fonttype":42,"svg.fonttype":"none"})
    navy,teal,orange="#203955","#008279","#AC4D23"
    fig,axs=plt.subplots(1,2,figsize=(8.2,2.9),layout="constrained")
    m=np.unique(np.round(np.geomspace(2,512,160))).astype(int); eta=np.exp(-np.log(m)-m*np.log(2.));r=1-1/m;q=1-r*r;t=(1-eta)/(1+r)
    axs[0].plot(m,t*np.sqrt(q)+eta,color=navy,label=r"Upper bound on $\sigma_2$")
    axs[0].plot(m,np.maximum(0,t*r-eta),color=orange,ls="--",label=r"Lower bound on $\delta_1$")
    axs[0].plot(m,np.maximum(0,t*np.sqrt((1+5*q*q)/6)-eta),color=teal,ls="-.",label=r"Lower bound on $\delta_2$")
    for width,col in [(16,orange),(64,teal)]:
        ell=np.arange(width+1)
        K=2.**width; length=width*K; prob=1./length
        ns=3+width*2.**ell
        xx=np.log(ns)/np.log(length)
        entropy=-xlogy(prob,prob)-xlogy(1-prob,1-prob)
        low=np.maximum(0,log(2)-np.log(ns)/width-entropy)
        high=(1-ell/width)*log(2)+1/width
        axs[1].fill_between(xx,low,high,color=col,alpha=.14)
        axs[1].plot(xx,high,color=col,lw=1,label=f"Finite interval, m={width}")
        axs[1].plot(xx,low,color=col,lw=1)
    axs[1].plot([0,1],[log(2),0],ls="--",color=navy,label="Exact limiting curve")
    axs[0].set_xscale("log",base=2);axs[0].set_xticks([2,8,32,128,512],labels=["2","8","32","128","512"])
    axs[0].set(xlabel="Block width  m",ylabel="Operator norm")
    axs[1].set(xlabel="State exponent  log N / log L",ylabel="Optimal excess log-loss (nats)",xlim=(0,1.03))
    for ax in axs:
        ax.grid(axis="y",alpha=.18);ax.legend(fontsize=7.5,frameon=False)
    fig.savefig(figures/"joint.pdf",metadata={"CreationDate":None,"ModDate":None});plt.close(fig)
    fig,ax=plt.subplots(figsize=(6.4,2.5),layout="constrained")
    for order,col,marker in [("ascending",navy,"o"),("descending",orange,"s")]:
        rr=[x for x in rows if x["order"]==order]
        ax.plot([x["m"] for x in rr],[x["logp"]/log(10) for x in rr],marker=marker,
                color=col,label=order.title())
    ax.plot(range(1,5),[bounds(i)["1"]/log(10) for i in range(1,5)],ls="--",color=teal,label="One-state cap")
    ax.set(xticks=[1,2,3,4],xlabel="Gate width in bits",ylabel=r"$\log_{10}$ completed-plan probability")
    ax.grid(axis="y",alpha=.18);ax.legend(ncol=3,fontsize=8,frameon=False)
    fig.savefig(figures/"planner.pdf",metadata={"CreationDate":None,"ModDate":None});plt.close(fig)
    table=[]
    for i in range(0,len(rows),2):
        a,b=rows[i:i+2];m=a["m"]
        table.append(f'{m} & {m*2**m} & {a["logp"]:.3f} & {b["logp"]:.3f} & {a["raw_logp"]:.3f} & {b["raw_logp"]:.3f} & {bounds(m)["1"]:.3f} '+r'\\')
    txt=r"""\begin{table}[ht]
\centering\small
\caption{Teacher-forced log probabilities in nats. Each target includes the final stop. The last column applies to every proper autonomous one-state controller.}\label{tab:qwen}
\begin{tabular}{rr rr rr r}\toprule
&& \multicolumn{2}{c}{Three-token policy}&\multicolumn{2}{c}{Full vocabulary}&\\
$m$ & Length & Asc. & Desc. & Asc. & Desc. & $\log c_1$\\\midrule
"""+ "\n".join(table)+r"""
\bottomrule\end{tabular}
\end{table}
"""
    txt+=r"""On the ascending two-action instance the constrained policy assigned the valid trace probability $0.4127$ (rounded). Its best one-state competitor can assign at most $1/27$. Equation~\eqref{eq:event} gives a floor of $0.7046$ nats, rounded down. The other seven one-state event tests, and all eight tests at budgets two and four, returned zero. None of the eight restricted greedy rollouts solved its instance, as determined by the first mismatch with the verified action sequence, including the stop. The full-vocabulary policy crossed none of the reported caps. A teacher-forced success probability and a greedy success indicator answer different questions.

On the two width-one prompts we also enumerated all completed traces under a three-action horizon. For ascending and descending order, the ratios $\sigma_2/\sigma_1$ were respectively """
    caps=[r['capped'] for r in rows if 'capped' in r]
    txt+=f"{caps[0]['sigma2']/caps[0]['sigma1']:.3f} and {caps[1]['sigma2']/caps[1]['sigma1']:.3f}."
    txt+=r""" In both cases the antisymmetric lower bound was below $\sigma_2$. These checkpoint observations do not establish the spectral separation proved for the constructed law."""
    txt=txt.replace("0.4127",format(exp(rows[0]["logp"]),".4f"))
    (paper/"measurements.tex").write_text(txt+"\n")
    details=r"""The action interface uses token IDs 362 (space-A), 425 (space-B), and 151645 (end of message). The ID order and complete chat prompt are checked by the released program. Each model call evaluates the target prefix causally; no search or target-specific logit mask is used. The measured sweep used the CPU execution provider of ONNX Runtime VERSION, three intra-operation threads and one inter-operation thread. The measured scoring time, after initializing the session, was TIME seconds. Only public model parameters and generated gate instances were used.
\begin{center}
\begin{tabular}{lrrrr}\toprule
Order at $m=1$ & $\sigma_1$ & $\sigma_2$ & $\frac12\|H-H^\top\|$ & Rank-gap witness\\\midrule
"""
    details=details.replace("VERSION",rec["versions"]["onnxruntime"]).replace("TIME",f'{rec["seconds"]:.1f}')
    for name,c in zip(("Ascending","Descending"),caps):
        details+=f'{name} & {c["sigma1"]:.4f} & {c["sigma2"]:.4f} & {c["skew_floor"]:.4f} & Inconclusive '+r'\\'+"\n"
    details+=r"\bottomrule\end{tabular}\end{center}"+"\n"
    (paper/"benchmark-details.tex").write_text(details)
    return len(rows)

if __name__=="__main__":
    import sys
    generate(sys.argv[1])

In [ ]:
from figures import generate
assert generate(ROOT)==8
from IPython.display import Image, display
for name in ("joint","planner"):
    pdf=ROOT/"paper/figures"/(name+".pdf")
    out=ROOT/"artifacts"/name
    subprocess.run(["pdftoppm","-png","-singlefile","-r","120",str(pdf),str(out)],check=True)
    display(Image(filename=str(out.with_suffix(".png"))))
for r in frozen["instances"]:
    print(f'{r["order"]:10s} width={r["m"]}  ln P={r["logp"]:8.3f}  one-state KL floor={r["kl_floor"]["1"]:.6f}')

## Compile and package
The ZIP has a single root document and includes the full supplement, vector figures and ancillary programs/data. The builder recompiles the outgoing ZIP from an empty directory and verifies the PDF rendering against the original. A standalone supplement and an anonymous review edition with the official checklist are also produced.

In [ ]:
%%writefile experiments/package.py
"""Build three PDFs and a single-entry-point arXiv source archive."""
from pathlib import Path
import os, shutil, subprocess, tempfile, zipfile, hashlib
from datetime import datetime, timezone

SOURCE_FILES=("main.tex","body.tex","preamble.tex","references.tex","proofs.tex",
              "measurements.tex","benchmark-details.tex","neurips_2026.sty",
              "figures/joint.pdf","figures/planner.pdf")

def compile_tex(source, target):
    source=Path(source);target=Path(target)
    env=dict(os.environ, SOURCE_DATE_EPOCH=str(int(datetime(2026,11,21,tzinfo=timezone.utc).timestamp())),
             FORCE_SOURCE_DATE="1",TZ="UTC")
    with tempfile.TemporaryDirectory() as tmp:
        tmp=Path(tmp)
        # One log per pass is kept in memory; print the tail on failure.
        for _ in range(3):
            res=subprocess.run(["pdflatex","-file-line-error","-interaction=nonstopmode","-halt-on-error",
                                "-output-directory="+str(tmp),source.name],cwd=source.parent,env=env,
                               stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors="replace")
            if res.returncode: raise RuntimeError(res.stdout[-6500:])
        log=(tmp/(source.stem+".log")).read_text(errors="replace")
        for marker in ("There were undefined references","There were undefined citations","multiply defined",
                       "Missing character:","Overfull \\hbox","Overfull \\vbox"):
            if marker in log: raise RuntimeError(f"{source.name}: {marker}")
        shutil.copyfile(tmp/(source.stem+".pdf"),target)
    return target

def write_zip(path, files):
    with zipfile.ZipFile(path,"w") as z:
        for arc,src in sorted(files.items()):
            item=zipfile.ZipInfo(arc,(2026,11,21,0,0,0));item.compress_type=zipfile.ZIP_DEFLATED
            item.external_attr=0o100644<<16
            z.writestr(item,Path(src).read_bytes())

def build(root,validate=True):
    root=Path(root).resolve(); p=root/"paper"; dest=root/"artifacts";dest.mkdir(exist_ok=True)
    for src,pdf in [('main.tex','DRAGA_Article.pdf'),('supplement-wrapper.tex','DRAGA_Mathematical_Supplement.pdf'),
                    ('review-wrapper.tex','DRAGA_Anonymous_Review.pdf')]:
        compile_tex(p/src,dest/pdf)
    arxiv=dest/"DRAGA_arXiv_Source.zip"
    include={name:p/name for name in SOURCE_FILES}
    include.update({"anc/qwen.json":root/"results/qwen.json",
                    "anc/planner.py":root/"experiments/planner.py",
                    "anc/figures.py":root/"experiments/figures.py",
                    "anc/REPRODUCING.txt":p/"ancillary-guide.txt"})
    write_zip(arxiv,include)
    if validate:
        # Rebuild using only files from the outgoing ZIP.
        with tempfile.TemporaryDirectory() as d:
            with zipfile.ZipFile(arxiv) as z: z.extractall(d)
            check=compile_tex(Path(d)/"main.tex",Path(d)/"rebuilt.pdf")
            if check.read_bytes()!=(dest/"DRAGA_Article.pdf").read_bytes():
                # pdfTeX's trailer ID may depend on the temporary location. Page rendering is invariant.
                def rendered(path,out):
                    subprocess.run(["pdftoppm","-r","40","-gray",str(path),str(out)],check=True)
                    return [hashlib.sha256(x.read_bytes()).hexdigest() for x in sorted(out.parent.glob(out.name+"-*.pgm"))]
                if rendered(check,Path(d)/"a")!=rendered(dest/"DRAGA_Article.pdf",Path(d)/"b"):
                    raise RuntimeError("Rebuilding the source ZIP changed the rendered pages")
    return dest

if __name__=="__main__":
    import sys
    for f in sorted(build(sys.argv[1]).iterdir()):print(f.name,f.stat().st_size)

In [ ]:
from package import build
OUTPUT=build(ROOT)
for path in sorted(OUTPUT.iterdir()):
    if path.suffix in (".pdf",".zip"):
        print(path.name,path.stat().st_size,"bytes")

## Download the deliverables
Upload **DRAGA_arXiv_Source.zip** through arXiv's TeX submission workflow. The article PDF is the reading copy. The notebook never submits anything automatically.

In [ ]:
from IPython.display import FileLink
os.chdir(OUTPUT)
for name in ("DRAGA_arXiv_Source.zip","DRAGA_Article.pdf","DRAGA_Mathematical_Supplement.pdf","DRAGA_Anonymous_Review.pdf"):
    display(FileLink(name))
if DOWNLOAD_ARXIV_ZIP and importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None:
    from google.colab import files
    files.download("DRAGA_arXiv_Source.zip")